# 01: データ準備

MedQuADデータセットをダウンロードして前処理します。

## MedQuADについて

- 医療Q&Aデータセット（NIHウェブサイトから作成）
- ライセンス: パブリックドメイン
- 約47,000件のQ&Aペア
- 12のNIHソース（Cancer.gov, GARD, NIDDK等）

## 1. ライブラリインポート

In [1]:
import os
import json
import requests
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm

print("インポート完了")

インポート完了


## 2. MedQuADデータをダウンロード

GitHubからXMLファイルをダウンロードしてJSONに変換します。

In [2]:
import subprocess

# git cloneでリポジトリをダウンロード
repo_url = "https://github.com/abachaa/MedQuAD.git"
local_path = "../temp_medquad"

# 既存の場合はスキップ
from pathlib import Path
if Path(local_path).exists():
    print(f"既に存在します: {Path(local_path).absolute()}")
else:
    print(f"クローン中: {repo_url}")
    subprocess.run(["git", "clone", repo_url, local_path], check=True)
    print(f"クローン完了: {Path(local_path).absolute()}")

既に存在します: C:\Users\hello\Desktop\project\PyTorch_Proj\portfolio\Medical-NLP\02-Medical-RAG\notebooks\..\temp_medquad


In [3]:
# クローンしたファイルを確認
print("=== クローン確認 ===")
print(f"ディレクトリ存在: {Path(local_path).exists()}")

if Path(local_path).exists():
    subdirs = [d for d in Path(local_path).iterdir() if d.is_dir()]
    print(f"サブディレクトリ数: {len(subdirs)}")
    
    # 最初のディレクトリから最初のXMLファイルを確認
    first_dir = subdirs[0]
    xml_files = list(first_dir.glob("*.xml"))
    if xml_files:
        print(f"\n[{first_dir.name}] ファイル数: {len(xml_files)}")
        
        # 最初のXMLファイルをパースして確認
        import xml.etree.ElementTree as ET
        with open(xml_files[0], 'r', encoding='utf-8') as f:
            content = f.read()
        root = ET.fromstring(content)
        
        focus = root.find('Focus')
        print(f"Focus: {focus.text if focus is not None else 'N/A'}")
        
        qapairs = root.find('QAPairs')
        if qapairs is not None and len(qapairs) > 0:
            qa = qapairs[0]
            q = qa.find('Question')
            a = qa.find('Answer')
            print(f"Q: {q.text[:60] if q is not None else 'N/A'}...")
            print(f"A: {a.text[:60] if a is not None and a.text else 'N/A'}...")

=== クローン確認 ===
ディレクトリ存在: True
サブディレクトリ数: 13


## 2. MedQuADデータをダウンロード

git cloneでリポジトリをローカルにコピーします。

In [4]:
# MedQuADのディレクトリ一覧（ローカルファイル読み込み）
MEDQUAD_DIRS = [
    "1_CancerGov_QA",
    "2_GARD_QA",
    "3_GHR_QA",
    "4_MPlus_Health_Topics_QA",
    "5_NIDDK_QA",
    "6_NINDS_QA",
    "7_SeniorHealth_QA",
    "8_NHLBI_QA_XML",
    "9_CDC_QA",
    "10_MPlus_ADAM_QA",
    "11_MPlusDrugs_QA",
    "12_MPlusHerbsSupplements_QA"
]

# ローカルgit cloneしたリポジトリを使用
LOCAL_MEDQUAD_PATH = "../temp_medquad"

print(f"対象ディレクトリ数: {len(MEDQUAD_DIRS)}")
print(f"ローカルパス: {Path(LOCAL_MEDQUAD_PATH).absolute()}")

対象ディレクトリ数: 12
ローカルパス: C:\Users\hello\Desktop\project\PyTorch_Proj\portfolio\Medical-NLP\02-Medical-RAG\notebooks\..\temp_medquad


## 3. データを収集

In [5]:
def fetch_medquad_data():
    """MedQuADデータをローカルファイルから取得"""
    all_qa_pairs = []
    
    for dir_name in MEDQUAD_DIRS:
        # ローカルディレクトリパス
        dir_path = Path(LOCAL_MEDQUAD_PATH) / dir_name
        
        if not dir_path.exists():
            print(f"  スキップ: {dir_name} (ディレクトリ不在)")
            continue
        
        # XMLファイルを取得
        xml_files = list(dir_path.glob("*.xml"))
        print(f"  {dir_name}: {len(xml_files)} ファイル")
        
        # XMLファイルをパース
        for xml_file in tqdm(xml_files, desc=f"    Processing", leave=False):
            try:
                with open(xml_file, 'r', encoding='utf-8') as f:
                    xml_content = f.read()
                
                root = ET.fromstring(xml_content)
                
                # Focus（トピック）を取得
                focus_elem = root.find('Focus')
                focus = focus_elem.text if focus_elem is not None else "Unknown"
                
                # QAPairsを取得
                qapairs_elem = root.find('QAPairs')
                if qapairs_elem is None:
                    continue
                
                for qa in qapairs_elem:
                    q_elem = qa.find('Question')
                    a_elem = qa.find('Answer')
                    
                    if q_elem is not None and a_elem is not None:
                        question = q_elem.text
                        answer = a_elem.text
                        
                        if question and answer:
                            all_qa_pairs.append({
                                "question": question,
                                "answer": answer,
                                "focus": focus,
                                "source": dir_name
                            })
            except Exception as e:
                print(f"Error parsing {xml_file.name}: {e}")
                continue
    
    return all_qa_pairs

# データを取得
print("=== MedQuADデータを取得中（ローカルファイル）===")
qa_pairs = fetch_medquad_data()
print(f"\n合計Q&Aペア数: {len(qa_pairs):,}")

=== MedQuADデータを取得中（ローカルファイル）===
  1_CancerGov_QA: 116 ファイル


  2_GARD_QA: 2685 ファイル


  3_GHR_QA: 1086 ファイル


  4_MPlus_Health_Topics_QA: 981 ファイル


  5_NIDDK_QA: 157 ファイル


...

  9_CDC_QA: 59 ファイル


  10_MPlus_ADAM_QA: 4366 ファイル


  11_MPlusDrugs_QA: 1312 ファイル


  12_MPlusHerbsSupplements_QA: 99 ファイル



合計Q&Aペア数: 16,407


## 4. データを確認

In [6]:
# サンプル確認
print("=== サンプルQ&A ===")
for i in range(min(3, len(qa_pairs))):
    qa = qa_pairs[i]
    print(f"\n[{i+1}] Focus: {qa['focus'][:60]}...")
    print(f"Q: {qa['question'][:80]}...")
    print(f"A: {qa['answer'][:100]}...")

=== サンプルQ&A ===

[1] Focus: Adult Acute Lymphoblastic Leukemia...
Q: What is (are) Adult Acute Lymphoblastic Leukemia ?...
A: Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in whi...

[2] Focus: Adult Acute Lymphoblastic Leukemia...
Q: What are the symptoms of Adult Acute Lymphoblastic Leukemia ?...
A: Signs and symptoms of adult ALL include fever, feeling tired, and easy bruising or bleeding. The ear...

[3] Focus: Adult Acute Lymphoblastic Leukemia...
Q: How to diagnose Adult Acute Lymphoblastic Leukemia ?...
A: Tests that examine the blood and bone marrow are used to detect (find) and diagnose adult ALL. The f...


## 5. RAG用フォーマットに変換

In [7]:
def format_for_rag(qa_pairs):
    """RAG用のドキュメントフォーマットに変換"""
    processed = []
    
    for qa in qa_pairs:
        # 質問と回答を組み合わせたドキュメントを作成
        content = f"Question: {qa['question']}\n\nAnswer: {qa['answer']}"
        
        processed.append({
            "content": content,
            "question": qa['question'],
            "focus": qa['focus'],
            "source": qa['source']
        })
    
    return processed

# 変換
processed_data = format_for_rag(qa_pairs)
print(f"変換完了: {len(processed_data)} ドキュメント")

変換完了: 16407 ドキュメント


## 6. 保存

In [8]:
# 保存先ディレクトリ
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

# JSON保存
output_file = data_dir / "medquad_processed.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(processed_data, f, ensure_ascii=False, indent=2)

print(f"保存完了: {output_file}")
print(f"ファイルサイズ: {output_file.stat().st_size / 1024 / 1024:.1f} MB")

保存完了: ..\data\medquad_processed.json
ファイルサイズ: 24.3 MB


---

**次**: `02_build_db.ipynb` でChromaDB構築